# Week 7: Evaluation Framework

Phase 1 마무리 - Pre-Freeze 평가

## 목표
1. **RAGAS 4지표 측정**: Faithfulness, Answer Relevancy, Context Precision, Context Recall
2. **도메인 특화 메트릭**: Refusal Accuracy, Citation Accuracy
3. **RAGAS 한계 사례 분석**: 점수와 실제 품질의 괴리 발굴

## 데이터셋
- Golden Set v1: `data/eval/golden_set_v1.csv` (35 questions, 5 q_types)

In [1]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.documents import Document

from src.evaluation import (
    EvalQuestion,
    load_golden_set,
    is_refusal,
    extract_citations,
    refusal_accuracy,
    citation_accuracy,
)
from src.agent import (
    make_filterable_hybrid_retriever,
    build_agent_graph,
    run_agent,
)


/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Golden Set v2

In [2]:
# load_golden_set + EvalQuestion live in src.evaluation (B1 SSOT consolidation).
# D1: switched to v2 (41q with image-required 8건 for cross-modal baseline).
GOLDEN_SET_PATH = Path('../data/eval/golden_set_v2.csv')
golden_questions = load_golden_set(GOLDEN_SET_PATH)
print(f"Loaded {len(golden_questions)} questions")

# Distribution by q_type + modality_label
q_type_counts = defaultdict(int)
modality_counts = defaultdict(int)
for q in golden_questions:
    q_type_counts[q.question_type] += 1
    modality_counts[q.modality_label] += 1
print(f"\nBy q_type: {dict(q_type_counts)}")
print(f"By modality: {dict(modality_counts)}")


Loaded 41 questions

By q_type: {'factual': 27, 'multi_hop': 4, 'comparison': 4, 'out_of_scope': 4, 'safety': 2}
By modality: {'text-only': 27, 'image-helpful': 6, 'image-required': 8}


## 2. Setup Baseline Retriever (Hybrid + Rerank)

In [3]:
from src.vectorstore import load_vectorstore
from src.retrieval import HybridRerankerRetriever, HybridRetrieverConfig, RerankConfig

CHROMA_DIR = Path('../data/chroma_db_c3')

print("Loading vectorstore...")
vs = load_vectorstore(CHROMA_DIR, collection_name="lg_manuals_c3")
print(f"Loaded {vs._collection.count()} documents")

print("\nCreating Hybrid+Rerank retriever...")
retriever = HybridRerankerRetriever(
    vs,
    hybrid_config=HybridRetrieverConfig(bm25_weight=0.5, dense_weight=0.5),
    rerank_config=RerankConfig(first_stage_k=20, final_k=5),
)
print("Retriever ready.")

Loading vectorstore...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded 258 documents

Creating Hybrid+Rerank retriever...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6782.29it/s]


Retriever ready.


## 3. RAG Pipeline with Citation

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

RAG_PROMPT = """다음 컨텍스트를 바탕으로 질문에 답하세요.
반드시 출처(문서명, 페이지)를 명시하세요.
컨텍스트에서 답을 찾을 수 없으면 "제공된 문서에서 확인할 수 없습니다."라고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:"""

def run_rag(question: str, retriever, llm) -> tuple[str, list[Document]]:
    """Run RAG pipeline with citation tracking."""
    docs = retriever.invoke(question)
    
    # Build context with source info
    context_parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'unknown')
        page = doc.metadata.get('page', '?')
        context_parts.append(f"[출처: {source} p.{page}]\n{doc.page_content}")
    
    context = "\n\n".join(context_parts)
    prompt = RAG_PROMPT.format(context=context, question=question)
    
    response = llm.invoke(prompt)
    return response.content, docs

# Test
test_q = golden_questions[0]
response, docs = run_rag(test_q.question, retriever, llm)
print(f"Q: {test_q.question}")
print(f"A: {response[:200]}...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Q: 정수기 필터 교체 주기는 얼마인가요?
A: 정수기 필터 교체 주기는 다음과 같습니다:
- 중금속9 흡착 필터: 6개월 (10 L/일 사용 기준)
- 바이러스 클리어 필터: 12개월 (10 L/일 사용 기준) 
(출처: waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf p.1)...


## 4. RAGAS 4지표 측정

In [5]:
# RAGAS 0.4.3+ modern API: instructor LLM + native ragas embeddings
# (The langchain LLM/Embeddings path is deprecated; ragas.metrics.collections
# metrics require InstructorBaseRagasLLM / BaseRagasEmbedding directly.)
from unittest.mock import MagicMock
_fake_vertexai = MagicMock()
_fake_vertexai.ChatVertexAI = MagicMock()
sys.modules["langchain_community.chat_models.vertexai"] = _fake_vertexai

import asyncio
import openai
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.metrics.collections import (
    Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall,
)

# One OpenAI client is reused by both the eval LLM and the eval embeddings.
_openai_client = openai.AsyncOpenAI()  # ragas .ascore() requires an async client
ragas_llm = llm_factory(
    model="gpt-4o-mini", provider="openai", client=_openai_client,
    max_tokens=8192,  # NLI verdict JSON for verbose Korean responses needs headroom
)
ragas_embeddings = RagasOpenAIEmbeddings(
    client=_openai_client, model="text-embedding-3-small",
)


In [6]:
import gc

async def evaluate_with_ragas(
    questions: list[EvalQuestion],
    retriever=None,
    llm=None,
    runner_fn=None,
    sample_size: int | None = None,
    score_concurrency: int = 3,
) -> pd.DataFrame:
    """Run RAGAS evaluation on golden set (modern collections API).

    `runner_fn(question) -> (answer: str, contexts: list[str])` lets us swap
    Baseline (run_rag) for Agentic (run_agent) on the same harness. If omitted,
    falls back to run_rag with `retriever` + `llm` (Baseline default).

    Memory guard: `score_concurrency` caps how many questions' 4-metric scoring
    runs in parallel. Default 3 keeps peak memory bounded — the previous
    unbounded `asyncio.gather` spawned 37x4=148 concurrent OpenAI requests and
    pushed RAM past 10GB on a 16GB machine.
    """
    eval_questions = [q for q in questions if q.q_type != 'out_of_scope']
    if sample_size:
        eval_questions = eval_questions[:sample_size]

    print(f"Evaluating {len(eval_questions)} questions...")

    results = []
    for i, q in enumerate(eval_questions):
        print(f"  [{i+1}/{len(eval_questions)}] {q.question[:30]}...")
        if runner_fn is not None:
            response, contexts = runner_fn(q.question)
        else:
            response, docs = run_rag(q.question, retriever, llm)
            contexts = [d.page_content for d in docs]
            del docs  # release Document refs immediately
        results.append({
            'question': q.question,
            'q_type': q.q_type,
            'ground_truth': q.ground_truth,
            'reference_context': q.reference_context,
            'response': response,
            'contexts': contexts,
        })
        if (i + 1) % 5 == 0:
            gc.collect()

    faithfulness = Faithfulness(llm=ragas_llm)
    answer_relevancy = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    context_precision = ContextPrecision(llm=ragas_llm)
    context_recall = ContextRecall(llm=ragas_llm)

    def _val(x):
        if isinstance(x, Exception):
            print(f"    metric failed: {type(x).__name__}: {str(x)[:80]}")
            return float('nan')
        return x.value

    sem = asyncio.Semaphore(score_concurrency)

    async def _score_one(r: dict) -> dict:
        async with sem:
            q, resp, ctxs, ref = r['question'], r['response'], r['contexts'], r['ground_truth']
            f, ar, cp, cr = await asyncio.gather(
                faithfulness.ascore(user_input=q, response=resp, retrieved_contexts=ctxs),
                answer_relevancy.ascore(user_input=q, response=resp),
                context_precision.ascore(user_input=q, reference=ref, retrieved_contexts=ctxs),
                context_recall.ascore(user_input=q, retrieved_contexts=ctxs, reference=ref),
                return_exceptions=True,
            )
            return {
                'faithfulness':       _val(f),
                'answer_relevancy':   _val(ar),
                'context_precision':  _val(cp),
                'context_recall':     _val(cr),
            }

    print(f"\nRunning RAGAS evaluation (concurrency={score_concurrency})...")
    scored = await asyncio.gather(*[_score_one(r) for r in results])

    merged = [{**r, **s} for r, s in zip(results, scored)]
    gc.collect()
    return pd.DataFrame(merged)


In [7]:
# Sprint 2 (D1): drop `sample_size` so the run covers the full golden set across
# all q_types (sprint 1 sampled 10 factual-only — non-representative).
ragas_results = await evaluate_with_ragas(golden_questions, retriever, llm)

# Summary statistics
print("\n=== RAGAS 4지표 Summary (Baseline, full golden set ex-out_of_scope) ===")
print(f"Faithfulness:       {ragas_results['faithfulness'].mean():.3f}")
print(f"Answer Relevancy:   {ragas_results['answer_relevancy'].mean():.3f}")
print(f"Context Precision:  {ragas_results['context_precision'].mean():.3f}")
print(f"Context Recall:     {ragas_results['context_recall'].mean():.3f}")

Evaluating 37 questions...
  [1/37] 정수기 필터 교체 주기는 얼마인가요?...
  [2/37] WD523A 모델의 제어창 사용법을 알려주세요...
  [3/37] 물맛이 이상할 때 어떻게 해야 하나요?...
  [4/37] 정수기 출수구 살균 기능은 어떻게 사용하나요?...
  [5/37] 온수 잠금 기능을 설정하는 방법...
  [6/37] AS281DAW 필터 수명은 얼마나 되나요?...
  [7/37] 공기청정기 필터 청소는 어떻게 하나요?...
  [8/37] 공기가 탁할 때 어떤 모드를 사용해야 하나요?...
  [9/37] 공기청정기 센서 청소 방법을 알려주세요...
  [10/37] 상태 표시등이 빨간색일 때 무슨 의미인가요?...
  [11/37] 청소기 배터리 충전 시간은 몇 시간인가요?...
  [12/37] 배터리가 빨리 닳아요...
  [13/37] 먼지 분리기 청소는 어떻게 하나요?...
  [14/37] 흡입구에 뭔가 걸렸을 때 어떻게 하나요?...
  [15/37] 보조 배터리 충전하는 방법...
  [16/37] LG ThinQ 앱 연결 방법을 알려주세요...
  [17/37] 와이파이 연결이 안 될 때...
  [18/37] 청소기 흡입력이 약해졌어요...
  [19/37] 필터 교체 후 해야 할 일이 있나요?...
  [20/37] 공기청정기 소음이 심해요...
  [21/37] WD325AS와 WD520AWB 정수기의 차이점은 무엇...
  [22/37] AS181DAW와 AS281DAW 공기청정기의 차이점은...
  [23/37] 정수 모드와 냉수 모드의 차이가 뭔가요?...
  [24/37] 유선 청소기와 무선 청소기의 장단점은?...
  [25/37] 공기청정기 필터 교체 후 필요한 초기화 과정은?...
  [26/37] 청소기 배터리 교체 후 최초 충전은 어떻게 하나요?...
  [27/37] 정수기 설치 후 처음 사용할 때 무엇을 확인해야 하나요...
  [28/37] 필터를 직접 분해해도

In [11]:
import torch
print("MPS available:", torch.backends.mps.is_available())
print("Reranker device:", getattr(reranker if 'reranker' in dir() else retriever.reranker, '_target_device',
    'unknown'))
if torch.backends.mps.is_available():
    print(f"MPS driver allocated: {torch.mps.driver_allocated_memory()/1024**3:.2f} GB")
    print(f"MPS current allocated: {torch.mps.current_allocated_memory()/1024**3:.2f} GB")

`CrossEncoder._target_device` has been deprecated. Please use `CrossEncoder.device` instead.


MPS available: True
Reranker device: mps:0
MPS driver allocated: 13.65 GB
MPS current allocated: 2.12 GB


In [13]:
import pickle
from pathlib import Path

ckpt = Path('../data/eval/_w7_checkpoint')
ckpt.mkdir(parents=True, exist_ok=True)

# Baseline RAGAS — 핵심 자산
ragas_results.to_pickle(ckpt / 'baseline_ragas_results.pkl')
print(f"✅ ragas_results saved: {ragas_results.shape}")

# 만약 cell 15·19도 어느 정도 돌았다면 같이 저장
for name in ['baseline_answers', 'refusal_result', 'citation_result']:
    if name in dir():
        with open(ckpt / f'{name}.pkl', 'wb') as f:
            pickle.dump(eval(name), f)
        print(f"✅ {name} saved")
    else:
        print(f"⏸ {name} not present (will redo)")


✅ ragas_results saved: (37, 10)
✅ baseline_answers saved
⏸ refusal_result not present (will redo)
⏸ citation_result not present (will redo)


In [7]:
import pandas as pd
ragas_results = pd.read_pickle('../data/eval/_w7_checkpoint/baseline_ragas_results.pkl')
print(f"Restored: {ragas_results.shape}")
print(f"Faithfulness: {ragas_results['faithfulness'].mean():.3f}")
print(f"Answer Relevancy: {ragas_results['answer_relevancy'].mean():.3f}")
print(f"Context Precision: {ragas_results['context_precision'].mean():.3f}")
print(f"Context Recall: {ragas_results['context_recall'].mean():.3f}")

Restored: (37, 10)
Faithfulness: 0.737
Answer Relevancy: 0.290
Context Precision: 0.847
Context Recall: 0.658


In [8]:
# By q_type breakdown
print("\n=== By q_type ===")
for q_type in ragas_results['q_type'].unique():
    subset = ragas_results[ragas_results['q_type'] == q_type]
    print(f"\n{q_type} (n={len(subset)}):")
    print(f"  Faithfulness:       {subset['faithfulness'].mean():.3f}")
    print(f"  Answer Relevancy: {subset['answer_relevancy'].mean():.3f}")
    print(f"  Context Precision:  {subset['context_precision'].mean():.3f}")
    print(f"  Context Recall:     {subset['context_recall'].mean():.3f}")


=== By q_type ===

factual (n=27):
  Faithfulness:       0.737
  Answer Relevancy: 0.294
  Context Precision:  0.883
  Context Recall:     0.716

multi_hop (n=4):
  Faithfulness:       0.844
  Answer Relevancy: 0.400
  Context Precision:  1.000
  Context Recall:     0.500

comparison (n=4):
  Faithfulness:       0.500
  Answer Relevancy: 0.184
  Context Precision:  0.375
  Context Recall:     0.250

safety (n=2):
  Faithfulness:       1.000
  Answer Relevancy: 0.236
  Context Precision:  1.000
  Context Recall:     1.000


## 5. Refusal Accuracy

In [9]:
# Refusal metric (is_refusal, refusal_accuracy, REFUSAL_PATTERNS) moved to src.evaluation.
# refusal_accuracy(questions, answers) decouples measurement from RAG execution so
# the same answers can be fed to RAGAS + Citation + Refusal in one pass (B2/B4).
#
# Example call (after running RAG on the full golden set):
#     answers = [run_rag(q.question, retriever, llm)[0] for q in golden_questions]
#     result = refusal_accuracy(golden_questions, answers)

In [9]:
# Replace the sprint-1 sample (`q.q_type in ['out_of_scope','safety']` + 5 factual)
# Sprint 2 (D1) — stratified Baseline + Agentic measurement.
# with the full golden set so the headline numbers are representative.
baseline_answers = []
for q in golden_questions:
    answer, _docs = run_rag(q.question, retriever, llm)
    baseline_answers.append(answer)

refusal_result = refusal_accuracy(golden_questions, baseline_answers)

print("=== Refusal Accuracy (Baseline, full golden set) ===")
print(f"Overall accuracy: {refusal_result['accuracy']:.1%}")
print(f"False-positive rate (answerable wrongly refused): {refusal_result['fp_rate']:.1%}")
print()
for q_type, stats in refusal_result['by_q_type'].items():
    print(f"  {q_type:<14} {stats['correct']}/{stats['total']}  acc={stats['accuracy']:.1%}")

=== Refusal Accuracy (Baseline, full golden set) ===
Overall accuracy: 80.5%
False-positive rate (answerable wrongly refused): 21.6%

  factual        22/27  acc=81.5%
  multi_hop      4/4  acc=100.0%
  comparison     1/4  acc=25.0%
  out_of_scope   4/4  acc=100.0%
  safety         2/2  acc=100.0%


In [10]:
# Already broken out by q_type in refusal_result['by_q_type'] above (cell 15). No
# separate aggregation needed — kept as a placeholder to preserve cell numbering.
# (Use `for q_type, stats in refusal_result['by_q_type'].items(): ...`.)

## 6. Citation Accuracy

In [11]:
# Citation metric moved to src.evaluation.
# - extract_citations(response) now handles the `..._OM_WEB.pdf p.1` format the
#   sprint-1 regex was missing (B3 fix; covered by tests/test_evaluation.py).
# - citation_accuracy(questions, answers) excludes out_of_scope from the
#   denominator and tolerates ±2 pages by default.

In [12]:
# Re-use the baseline_answers computed in cell 15 — no need to re-run the RAG pipeline.
citation_result = citation_accuracy(golden_questions, baseline_answers)

print("=== Citation Accuracy (Baseline, full golden set) ===")
print(f"Overall accuracy: {citation_result['accuracy']:.1%} ({citation_result['n']} q examined)")
print()
for q_type, stats in citation_result['by_q_type'].items():
    print(f"  {q_type:<14} {stats['correct']}/{stats['total']}  acc={stats['accuracy']:.1%}")

# Sanity: a sample answer + the citations extract pulls out
sample_idx = 0
sample_answer = baseline_answers[sample_idx]
print(f"\nSample [{sample_idx}] answer head: {sample_answer[:120]}...")
print(f"Extracted citations: {extract_citations(sample_answer)}")

=== Citation Accuracy (Baseline, full golden set) ===
Overall accuracy: 2.7% (37 q examined)

  factual        0/27  acc=0.0%
  multi_hop      0/4  acc=0.0%
  comparison     0/4  acc=0.0%
  safety         1/2  acc=50.0%

Sample [0] answer head: 정수기 필터 교체 주기는 다음과 같습니다:
- 중금속9 흡착 필터: 6개월 (10 L/일 사용 기준)
- 바이러스 클리어 필터: 12개월 (10 L/일 사용 기준) 
(출처: waterpurifier_simple_W...
Extracted citations: [('waterpurifier_simple', 1)]


## 6.5 Agentic Measurement (W6 모듈, D1 — TODO 채우기)

`src.agent.run_agent`로 Baseline과 같은 골든셋·같은 지표(RAGAS 4 + Refusal + Citation)를 측정해
`agentic_block`을 만들고 cell 24의 `results_summary['agentic'] = agentic_block`을 enable한다.

대략적인 셸 (cell 5의 vectorstore/all_docs/reranker, cell 7의 llm을 재사용):

```python
from src.vectorstore import load_vectorstore
from src.retrieval import extract_documents_from_vectorstore, create_reranker

# cell 5의 vectorstore/all_docs/reranker가 이미 메모리에 있으면 재사용.
retriever_fn = make_filterable_hybrid_retriever(vs, extract_documents_from_vectorstore(vs), retriever.reranker)
app = build_agent_graph(retriever_fn, llm)

agentic_answers = []
agentic_contexts = []
for q in golden_questions:
    r = run_agent(app, q.question)
    agentic_answers.append(r['answer'])
    agentic_contexts.append([d.page_content for d in r['documents']])

# RAGAS는 evaluate_with_ragas의 runner 인자화가 필요 — 다음 라인은 cell 10 리팩터를 전제로 한다.
# agentic_ragas = await evaluate_with_ragas(golden_questions, runner=lambda q: (answer, contexts), llm)
# agentic_refusal = refusal_accuracy(golden_questions, agentic_answers)
# agentic_citation = citation_accuracy(golden_questions, agentic_answers)
```

> Note: `evaluate_with_ragas`가 `runner_fn`을 받도록 cell 10을 작게 리팩터한 뒤 Agentic 측정을 켠다.
> 이번 sprint 산출물 §D1·D2~D4·E의 입력값이 된다.

In [13]:
# D1 Agentic measurement — same golden set, same metrics, run_agent runner.
from src.retrieval import extract_documents_from_vectorstore

all_docs = extract_documents_from_vectorstore(vs)
retriever_fn_a = make_filterable_hybrid_retriever(vs, all_docs, retriever.reranker)
app = build_agent_graph(retriever_fn_a, llm)

print(f"Running Agentic ({len(golden_questions)}q)...")
agentic_answers = []
agentic_contexts = []
agentic_route = []
agentic_retry = []
agentic_latency = []
for i, q in enumerate(golden_questions):
    r = run_agent(app, q.question)
    agentic_answers.append(r['answer'])
    agentic_contexts.append([d.page_content for d in r['documents']])
    agentic_route.append(r.get('route_history', []))
    agentic_retry.append(r.get('retry_count', 0))
    agentic_latency.append(r.get('total_latency', 0.0))
    print(f"  [{i+1}/{len(golden_questions)}] retry={r.get('retry_count', 0)} {r.get('total_latency', 0):.1f}s — {q.question[:35]}...")

print(f"\nAgentic mean latency: {sum(agentic_latency)/len(agentic_latency):.2f}s")
print(f"Retry distribution: {dict(Counter(agentic_retry))}")


Running Agentic (41q)...
  [1/41] retry=0 23.8s — 정수기 필터 교체 주기는 얼마인가요?...
  [2/41] retry=0 25.2s — WD523A 모델의 제어창 사용법을 알려주세요...
  [3/41] retry=0 33.8s — 물맛이 이상할 때 어떻게 해야 하나요?...
  [4/41] retry=0 23.9s — 정수기 출수구 살균 기능은 어떻게 사용하나요?...
  [5/41] retry=0 24.1s — 온수 잠금 기능을 설정하는 방법...
  [6/41] retry=1 37.5s — AS281DAW 필터 수명은 얼마나 되나요?...
  [7/41] retry=2 58.8s — 공기청정기 필터 청소는 어떻게 하나요?...
  [8/41] retry=0 25.0s — 공기가 탁할 때 어떤 모드를 사용해야 하나요?...
  [9/41] retry=0 27.9s — 공기청정기 센서 청소 방법을 알려주세요...
  [10/41] retry=0 25.3s — 상태 표시등이 빨간색일 때 무슨 의미인가요?...
  [11/41] retry=0 25.6s — 청소기 배터리 충전 시간은 몇 시간인가요?...
  [12/41] retry=0 31.5s — 배터리가 빨리 닳아요...
  [13/41] retry=0 26.8s — 먼지 분리기 청소는 어떻게 하나요?...
  [14/41] retry=0 25.7s — 흡입구에 뭔가 걸렸을 때 어떻게 하나요?...
  [15/41] retry=0 26.6s — 보조 배터리 충전하는 방법...
  [16/41] retry=0 19.7s — LG ThinQ 앱 연결 방법을 알려주세요...
  [17/41] retry=0 24.3s — 와이파이 연결이 안 될 때...
  [18/41] retry=0 28.3s — 청소기 흡입력이 약해졌어요...
  [19/41] retry=0 24.5s — 필터 교체 후 해야 할 일이 있나요?...
  [20/41] retry=2 655.8s — 공기청정

In [14]:
# Wrap per-question (answer, contexts) as a runner_fn so evaluate_with_ragas
# can score Agentic on the same harness as Baseline.
agentic_pairs = {q.question: (a, c) for q, a, c in zip(golden_questions, agentic_answers, agentic_contexts)}

def agentic_runner(question: str):
    return agentic_pairs[question]

agentic_ragas = await evaluate_with_ragas(golden_questions, runner_fn=agentic_runner)

print("\n=== RAGAS 4지표 Summary (Agentic, full golden set ex-out_of_scope) ===")
print(f"Faithfulness:       {agentic_ragas['faithfulness'].mean():.3f}")
print(f"Answer Relevancy:   {agentic_ragas['answer_relevancy'].mean():.3f}")
print(f"Context Precision:  {agentic_ragas['context_precision'].mean():.3f}")
print(f"Context Recall:     {agentic_ragas['context_recall'].mean():.3f}")


Evaluating 37 questions...
  [1/37] 정수기 필터 교체 주기는 얼마인가요?...
  [2/37] WD523A 모델의 제어창 사용법을 알려주세요...
  [3/37] 물맛이 이상할 때 어떻게 해야 하나요?...
  [4/37] 정수기 출수구 살균 기능은 어떻게 사용하나요?...
  [5/37] 온수 잠금 기능을 설정하는 방법...
  [6/37] AS281DAW 필터 수명은 얼마나 되나요?...
  [7/37] 공기청정기 필터 청소는 어떻게 하나요?...
  [8/37] 공기가 탁할 때 어떤 모드를 사용해야 하나요?...
  [9/37] 공기청정기 센서 청소 방법을 알려주세요...
  [10/37] 상태 표시등이 빨간색일 때 무슨 의미인가요?...
  [11/37] 청소기 배터리 충전 시간은 몇 시간인가요?...
  [12/37] 배터리가 빨리 닳아요...
  [13/37] 먼지 분리기 청소는 어떻게 하나요?...
  [14/37] 흡입구에 뭔가 걸렸을 때 어떻게 하나요?...
  [15/37] 보조 배터리 충전하는 방법...
  [16/37] LG ThinQ 앱 연결 방법을 알려주세요...
  [17/37] 와이파이 연결이 안 될 때...
  [18/37] 청소기 흡입력이 약해졌어요...
  [19/37] 필터 교체 후 해야 할 일이 있나요?...
  [20/37] 공기청정기 소음이 심해요...
  [21/37] WD325AS와 WD520AWB 정수기의 차이점은 무엇...
  [22/37] AS181DAW와 AS281DAW 공기청정기의 차이점은...
  [23/37] 정수 모드와 냉수 모드의 차이가 뭔가요?...
  [24/37] 유선 청소기와 무선 청소기의 장단점은?...
  [25/37] 공기청정기 필터 교체 후 필요한 초기화 과정은?...
  [26/37] 청소기 배터리 교체 후 최초 충전은 어떻게 하나요?...
  [27/37] 정수기 설치 후 처음 사용할 때 무엇을 확인해야 하나요...
  [28/37] 필터를 직접 분해해도

In [15]:
# Agentic Refusal + Citation (same metrics, same golden set)
agentic_refusal = refusal_accuracy(golden_questions, agentic_answers)
agentic_citation = citation_accuracy(golden_questions, agentic_answers)

print("=== Refusal Accuracy (Agentic) ===")
print(f"Overall: {agentic_refusal['accuracy']:.1%}  FP rate: {agentic_refusal['fp_rate']:.1%}")
for q_type, stats in agentic_refusal['by_q_type'].items():
    print(f"  {q_type:<14} {stats['correct']}/{stats['total']}  acc={stats['accuracy']:.1%}")

print("\n=== Citation Accuracy (Agentic) ===")
print(f"Overall: {agentic_citation['accuracy']:.1%} ({agentic_citation['n']} q)")
for q_type, stats in agentic_citation['by_q_type'].items():
    print(f"  {q_type:<14} {stats['correct']}/{stats['total']}  acc={stats['accuracy']:.1%}")


=== Refusal Accuracy (Agentic) ===
Overall: 75.6%  FP rate: 27.0%
  factual        22/27  acc=81.5%
  multi_hop      4/4  acc=100.0%
  comparison     0/4  acc=0.0%
  out_of_scope   4/4  acc=100.0%
  safety         1/2  acc=50.0%

=== Citation Accuracy (Agentic) ===
Overall: 0.0% (37 q)
  factual        0/27  acc=0.0%
  multi_hop      0/4  acc=0.0%
  comparison     0/4  acc=0.0%
  safety         0/2  acc=0.0%


## 7. RAGAS 한계 사례 분석

In [16]:
# Find cases where RAGAS score and actual quality diverge

def analyze_ragas_limits(ragas_df: pd.DataFrame) -> dict:
    """Find RAGAS limitation cases."""
    limits = {
        'high_score_bad_answer': [],  # RAGAS high but actually bad
        'low_score_good_answer': [],  # RAGAS low but actually good
    }
    
    for _, row in ragas_df.iterrows():
        faith = row['faithfulness']
        relevancy = row['answer_relevancy']
        
        # High score but potentially bad answer
        # Look for cases where faithfulness is high but answer is short/incomplete
        response_len = len(row['response'])
        if faith > 0.8 and response_len < 50:
            limits['high_score_bad_answer'].append({
                'question': row['question'],
                'response': row['response'],
                'faithfulness': faith,
                'reason': 'High faithfulness but very short response - may miss details',
            })
        
        # Low score but potentially good answer
        if faith < 0.5 and response_len > 100:
            limits['low_score_good_answer'].append({
                'question': row['question'],
                'response': row['response'],
                'faithfulness': faith,
                'reason': 'Low faithfulness but detailed response - may be over-penalized',
            })
    
    return limits

In [17]:
# Analyze RAGAS limits from the evaluation results
if 'ragas_results' in dir() and len(ragas_results) > 0:
    limits = analyze_ragas_limits(ragas_results)
    
    print("=== RAGAS 한계 사례 ===")
    print(f"\n점수 높은데 답변 나쁜 사례: {len(limits['high_score_bad_answer'])}건")
    for case in limits['high_score_bad_answer'][:2]:
        print(f"  Q: {case['question'][:50]}...")
        print(f"  A: {case['response'][:80]}...")
        print(f"  Faithfulness: {case['faithfulness']:.2f}")
        print(f"  분석: {case['reason']}")
        print()
    
    print(f"\n점수 낮은데 답변 괜찮은 사례: {len(limits['low_score_good_answer'])}건")
    for case in limits['low_score_good_answer'][:2]:
        print(f"  Q: {case['question'][:50]}...")
        print(f"  A: {case['response'][:80]}...")
        print(f"  Faithfulness: {case['faithfulness']:.2f}")
        print(f"  분석: {case['reason']}")
        print()
else:
    print("RAGAS 결과가 없습니다. 먼저 섹션 4를 실행하세요.")

=== RAGAS 한계 사례 ===

점수 높은데 답변 나쁜 사례: 1건
  Q: 필터 두 개 중 왼쪽 자리(①)에는 어떤 필터를 끼우나요?...
  A: 제공된 문서에서 확인할 수 없습니다....
  Faithfulness: 1.00
  분석: High faithfulness but very short response - may miss details


점수 낮은데 답변 괜찮은 사례: 2건
  Q: 공기가 탁할 때 어떤 모드를 사용해야 하나요?...
  A: 공기가 탁할 때는 "오토모드"를 사용해야 합니다. 오토모드는 종합청정도 단계에 따라 운전모드와 청정세기를 자동으로 조절하여 공기를 정화합니다. ...
  Faithfulness: 0.33
  분석: Low faithfulness but detailed response - may be over-penalized

  Q: 청소기 흡입력이 약해졌어요...
  A: 흡입력이 약해졌다면 다음의 원인 및 해결책을 확인하세요:

1. 먼지통이 가득 차 있습니다. • 먼지통을 비우세요.
2. 필터가 막혀 있습니다....
  Faithfulness: 0.09
  분석: Low faithfulness but detailed response - may be over-penalized



## 8. Save Results

In [18]:
# Save evaluation results — both Baseline and Agentic, with q_type AND
# modality_label breakdowns so the retrospective §9 (modality decomposition)
# can read it back without re-measuring.
RESULTS_PATH = Path('../data/eval/week7_results.json')
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)


def _modality_split(df: pd.DataFrame, label_lookup: dict) -> dict:
    df = df.copy()
    df['modality_label'] = df['question'].map(label_lookup).fillna('text-only')
    out = {}
    for label, sub in df.groupby('modality_label'):
        out[label] = {
            'n': int(len(sub)),
            'faithfulness':      float(sub['faithfulness'].mean()),
            'answer_relevancy':  float(sub['answer_relevancy'].mean()),
            'context_precision': float(sub['context_precision'].mean()),
            'context_recall':    float(sub['context_recall'].mean()),
        }
    return out


def _q_type_split(df: pd.DataFrame) -> dict:
    out = {}
    for qt, sub in df.groupby('q_type'):
        out[qt] = {
            'n': int(len(sub)),
            'faithfulness':      float(sub['faithfulness'].mean()),
            'answer_relevancy':  float(sub['answer_relevancy'].mean()),
            'context_precision': float(sub['context_precision'].mean()),
            'context_recall':    float(sub['context_recall'].mean()),
        }
    return out


label_lookup = {q.question: q.modality_label for q in golden_questions}

baseline_block = {
    'ragas_aggregate': {
        'faithfulness':      float(ragas_results['faithfulness'].mean()),
        'answer_relevancy':  float(ragas_results['answer_relevancy'].mean()),
        'context_precision': float(ragas_results['context_precision'].mean()),
        'context_recall':    float(ragas_results['context_recall'].mean()),
    },
    'ragas_by_q_type': _q_type_split(ragas_results),
    'ragas_by_modality': _modality_split(ragas_results, label_lookup),
    'refusal': refusal_result,
    'citation': citation_result,
}

agentic_block = {
    'ragas_aggregate': {
        'faithfulness':      float(agentic_ragas['faithfulness'].mean()),
        'answer_relevancy':  float(agentic_ragas['answer_relevancy'].mean()),
        'context_precision': float(agentic_ragas['context_precision'].mean()),
        'context_recall':    float(agentic_ragas['context_recall'].mean()),
    },
    'ragas_by_q_type': _q_type_split(agentic_ragas),
    'ragas_by_modality': _modality_split(agentic_ragas, label_lookup),
    'refusal': agentic_refusal,
    'citation': agentic_citation,
    'latency_mean_s': float(sum(agentic_latency)/len(agentic_latency)),
    'retry_distribution': dict(Counter(agentic_retry)),
}

results_summary = {
    'golden_set': {
        'path': str(GOLDEN_SET_PATH),
        'n': len(golden_questions),
        'by_q_type': dict(q_type_counts),
        'by_modality': dict(modality_counts),
    },
    'baseline': baseline_block,
    'agentic': agentic_block,
}

with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False, default=str)

print(f"Results saved to {RESULTS_PATH}")
print(f"Baseline RAGAS aggregate: {baseline_block['ragas_aggregate']}")
print(f"Agentic  RAGAS aggregate: {agentic_block['ragas_aggregate']}")


Results saved to ../data/eval/week7_results.json
Baseline RAGAS aggregate: {'faithfulness': 0.7373347373347374, 'answer_relevancy': 0.2901108744275262, 'context_precision': 0.8471846846594884, 'context_recall': 0.6576576576576577}
Agentic  RAGAS aggregate: {'faithfulness': 0.759810405643739, 'answer_relevancy': 0.2951301438878284, 'context_precision': 0.8266516516263956, 'context_recall': 0.7387387387387387}


In [ ]:
# T1-4: Citation-only re-measurement with MPS reranker (post-rebuild).
# Cheap path: re-runs run_rag_pipeline on the full golden set, computes
# citation_accuracy only (no RAGAS). Validates the by_page=True + prompt-tag fix.
import time
from pathlib import Path
from src.vectorstore import load_vectorstore
from src.retrieval import HybridRerankerRetriever, create_reranker
from src.evaluation import (
    load_golden_set, run_rag_pipeline,
    citation_accuracy, extract_citations,
)
from langchain_openai import ChatOpenAI

vs = load_vectorstore(Path("../data/chroma_db_c3"), collection_name="lg_manuals_c3")
retriever = HybridRerankerRetriever(vs)
# Swap CPU reranker → MPS
retriever.reranker = create_reranker(retriever.rerank_config.model_name, device="mps")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

questions = load_golden_set(Path("../data/eval/golden_set_v2.csv"))
print(f"Loaded {len(questions)} questions")

answers, latencies = [], []
t0 = time.time()
for i, q in enumerate(questions):
    s = time.time()
    ans, _ = run_rag_pipeline(q.question, retriever, llm, k=5)
    latencies.append(time.time() - s)
    answers.append(ans)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(questions)}  avg {sum(latencies)/len(latencies):.2f}s/q")
print(f"Total {time.time()-t0:.1f}s  avg {sum(latencies)/len(latencies):.2f}s/q")

cit = citation_accuracy(questions, answers, page_tolerance=2)
print(f"\nCitation Accuracy: {cit['accuracy']:.3f} ({cit['correct']}/{cit['scored']})  unscored={cit['unscored']}")

# Per-question table for eyeball check
print("\n#   q_type        expected                    cited                       match")
print("-" * 100)
for q, a in zip(questions, answers):
    cites = extract_citations(a)
    exp = q.reference_context[:30] if q.reference_context else "(none)"
    cit_str = ", ".join(f"{m} p.{p}" for m, p in cites) or "(no citation)"
    print(f"{q.id:>3} {q.q_type:13s} {exp:30s} {cit_str[:40]:40s}")


## Summary

### 1. 최종 점수 (sample_size=10, all `factual`)

| Metric | Score | 판단 |
|---|---|---|
| Faithfulness | **0.842** | ✅ 임계 ≥0.8 통과. 응답이 컨텍스트에 충실. |
| Answer Relevancy | **0.347** | ⚠️ 임계 한참 미달. **점수 자체보다 측정 신뢰성을 의심** (아래 §4 참고). |
| Context Precision | **0.901** | ✅ 임계 통과. 검색된 청크가 reference에 잘 기여. |
| Context Recall | **0.700** | ⚠️ 임계 ≥0.8 미달. reference 일부가 검색 컨텍스트로 직접 뒷받침 안 됨. |
| Refusal Accuracy | **1.000** | ✅ out_of_scope 4/4 + safety 2/2 + factual 5/5 모두 정답. |
| Citation Accuracy | **0.000** | ❌ **cell 18 regex 버그** — 실제 인용 품질이 아님 (아래 §4 참고). |

> Raw JSON: `data/week7_evaluation_results.json` (cell 24 출력).

---

### 2. RAGAS 4지표 정리

| 지표 | 무엇을 보는가 | 계산 방식 | 실용 임계 | 알려진 한계 |
|---|---|---|---|---|
| **Faithfulness** | 응답 ↔ 컨텍스트 | 응답을 LLM이 statement 단위로 분해 → 각 statement가 컨텍스트로 NLI 추론 가능한지 판정 → 지지된 비율 | ≥ 0.8 | (1) 짧은 응답은 statement 수가 적어 통계가 불안정. (2) 의역·요약을 NLI가 종종 over-penalize. |
| **Answer Relevancy** | 응답 ↔ 질문 | 응답으로부터 LLM이 N=3개의 가상 질문 생성 → 임베딩 → 원래 user_input과 평균 cosine 유사도 | ≥ 0.7 | (1) 거절 응답은 의도와 무관하게 점수 낮음. **(2) 한국어 임베딩 품질에 직접 의존 — 이번 결과의 0.347 원인 추정.** |
| **Context Precision** | 컨텍스트 ↔ reference | 검색된 청크들이 reference 도출에 기여하는지 LLM이 청크별로 판정 → MAP 형태 가중 평균 | ≥ 0.8 | (1) reference가 짧으면 다수 청크가 "무관"으로 판정되어 보수적. (2) 중복 청크가 후순위면 0 처리되어 평균 하락. |
| **Context Recall** | reference ↔ 컨텍스트 | reference 텍스트를 statement로 분해 → 각 statement가 검색된 컨텍스트로 뒷받침되는지 비율 | ≥ 0.8 | (1) reference에 추론·요약 진술이 섞이면 직접 뒷받침이 어려워 점수 낮음. (2) reference 자체 품질·세분성에 강하게 의존. |

---

### 3. 도메인 특화 지표 정리

| 지표 | 정의 | 실용 임계 | 측정 의의 |
|---|---|---|---|
| **Refusal Accuracy** | `out_of_scope` 질문을 거절(REFUSAL_PATTERNS 매칭)하고 answerable 질문에는 답하는 binary correctness 평균 | ≥ 0.9 | hallucination 차단 능력. **이번에 100% — 매우 강건.** |
| **Citation Accuracy** | 응답에 추출된 (source, page) 인용이 reference_context와 ±2 페이지 이내 일치하는 비율. `should_cite=True`만 분모. | ≥ 0.7 | 사용자가 답변의 근거를 직접 검증 가능한지. 가전 매뉴얼 도메인에서 특히 중요. |

---

### 4. 점수와 실제 품질의 괴리 — Phase 1 종합 분석

이번 run에서 cell 22의 자동 휴리스틱(`analyze_ragas_limits`)은 0건을 잡았지만, **육안으로 보면 두 가지 명백한 괴리가 있음:**

#### 4-1. Answer Relevancy 0.347은 모델 품질이 아니라 측정 한계

- Faithfulness 0.842, Context Precision 0.901 → 응답이 컨텍스트로 잘 뒷받침되고, 검색도 정확.
- 그런데 Answer Relevancy만 0.347 → **불일치는 측정 도구 쪽 문제일 가능성이 큼**.
- 가설: 한국어 응답 → 한국어 가상 질문 생성 → `text-embedding-3-small`로 임베딩 → 원 질문과 cosine. `text-embedding-3-small`은 한국어에 최적화돼 있지 않아 동일 의미 문장 간 cosine이 낮게 나오는 경향.
- **검증 액션 (next week):** 동일 데이터셋에서 임베딩만 `text-embedding-3-large` 또는 다국어 모델(`bge-m3`)로 바꿔 같은 metric 재측정 → 점수 상승 폭으로 측정 신뢰성 평가.

#### 4-2. Citation Accuracy 0%는 모델이 아니라 평가 코드 버그

- 응답은 실제로 출처를 인용함 (cell 7 샘플: `[출처: waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf p.1]`).
- cell 18의 정규식 `((?:waterpurifier|airpurifier|vacuumcleaner)_\w+)\s*p?\.?(\d+)`은 `_\w+` 다음에 곧장 페이지를 기대하는데, 실제로는 `.pdf p.1`이 중간에 들어가 매칭 실패.
- **검증 액션:** regex를 `((?:waterpurifier|airpurifier|vacuumcleaner)_\w+)[^\d]*?p\.?\s*(\d+)` 정도로 수정 후 재측정 → 실제 인용 정확도 산출.

#### 4-3. 이번 sample이 전부 `factual` (sample_size=10 우연)

- 골든셋 36문항 중 factual이 22문항이라 앞 10개가 다 factual로 잡힘.
- **multi_hop, comparison, safety는 평가되지 않음.** Phase 1 종결 판단을 하려면 stratified sampling으로 q_type별 최소 N개 평가 필요.

---

### 5. Phase 1 → Phase 2 시사점

- **텍스트 RAG는 factual에서 baseline 통과**: Faithfulness/Precision은 임계 위, Refusal 100%.
- **Context Recall 0.700**은 reference 분해 시 일부 statement가 retrieved chunk로 직접 뒷받침되지 않는다는 신호 → Phase 2 cross-modal 도입으로 그림 기반 절차 진술을 채우면 개선 여지.
- **RAGAS raw score만으로 판단 금지** 케이스 확인:
  - 거절 응답 → Answer Relevancy 점수 낮음 (의도 무시)
  - 한국어 응답 → Answer Relevancy 자체가 임베딩 품질에 끌려 내려감 (이번 결과)
  - reference가 여러 페이지에 분산 → Context Recall 분해가 어려워 보수적 점수
- **Phase 2 시작 전 권장사항** (ADR-007 평가 신뢰성):
  1. Golden Set v2: q_type별 stratified, reference_context 페이지 단위 세분화
  2. 임베딩 모델 한국어 검증 (text-embedding-3-large vs bge-m3 비교)
  3. Citation regex 수정 + 실제 인용 정확도 재측정
